# Image processing notebook: From overlap corrected to transmission 

### 00 - exp 1XX acquisition 00

##  Initial settings

### Import libraries
Import all the required libraries

In [1]:
import sys
sys.path.append(r'..\01_Functions')
from step_functions import *
from dict_functions import *
from proc_functions import *
from img_functions import *
%matplotlib inline

### Select directories
Select the source directory. This directory is where the images **after** the overlap correction were saved.
Select the destination directory. Here is where the transmission images are going to be saved.

In [2]:
# %load select_directory('src_dir')
src_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\00_Overlap_correction\exp1XX"

In [3]:
# %load select_directory('dst_dir')
dst_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\01_New_transmission_results"

### Selec folders to process

In [4]:
stack_dict = prep_stack_dict(src_dir)
for key in stack_dict.keys():
    print(key)

00_ob
01_so_ref
02_exp102_00
02_exp102_01
02_exp102_02
02_exp102_03
02_exp102_04
03_ob_end


In [5]:
ref_folder = ['01_so_ref']
proc_folder = ['02_exp102_00']

## TEST processing for reference images

### Build the reference images dictionary 
keep_acq_numb: amount of acquisition folders to process. this is a test so we want to so it fast. thus, we limit the amount of data <br>
In case you want to get an image _(array, header)_ in the dictionary, follow the format:<br>

`test = extract_img_dict(exp_test_dict, proc_folder[0], img_number = 50)`<br>
"variable" = function("dictionary name", "key/folder", "acq_number = 0", "img_number = 50")

**_To visualize the image_** (a tuple with an array in position 0 and a header in position 1, you require the next instruction:<br>

`show_img(test[0])
test[1]` # to show the header of that image

In [6]:
ref_test_dict, ref_test_param = testing_mode_step (src_dir, proc_folder = ref_folder, keep_acq_numb = 10)

Reading Images: 100%|████████████████████████████| 3/3 [00:13<00:00,  4.42s/it]


### Pre-processing sequence with parameters
The pre-processing sequence refers to any sequence with their respective parameters that emphasize on modifying or enhancing the image (**usually filters**). In this step you can also perform an averaging of all the acquisitions `stack_averaging`, a step binning of the aquisitions `binning_acquisitions`, and/or a step binning of the frames `binning_frames`.

**The image processing (SBKG, registration, scrubbing, and/or TFC should be done in the processing step.**

**NOTE: The order of the functions in the sequence matters**, but not the order of the parameters, you can also include alien parameters, this will not affect the process.

In [7]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(ref_test_param,['threshold'], [0])

In [ ]:
ref_test_dict = pre_processing_step (ref_test_dict, pre_proc_seq, param_dict = ref_test_param)

### Processing sequence and variables to obtain the referenceimages used in the TFC

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
add_to_dict(ref_test_param,['BB_mask'], [BB_mask])

In [ ]:
ref_test_dict = processing_step (ref_test_dict, proc_seq, param_dict = ref_test_param)

In [ ]:
proc_seq_save_ref = [TFC_correction_dict]
param_save_ref = {}
add_to_dict(param_save_ref,['nca', 'use_ref', 'ref_dict'], [nca, False, ref_dict])
ref_dict_save_ref = processing_step (ref_dict, proc_seq = proc_seq_save_ref, param_dict = param_save_ref)

### Get NCA
There will be displacements in the experiment images, for this reason, it is better to get the `nca` from the reference image.

In [ ]:
# %load select_rois(ref_img_TFC[0], list_rois = ['nca'])
nca = [408, 399, 43, 10]

In [ ]:
ref_img = extract_img_dict(ref_test_dict, ref_folder[0], img_number = 50)
show_img_rois(ref_img[0], dr = [(nca, 'blue')])

In [ ]:
dst_dir_test = dst_dir + '/registration_ref_test'
saving_step (ref_test_dict, dst_dir_test, img_name = 'ref')

## TEST processing for experiment images

In [ ]:
exp_test_dict, exp_test_param = testing_mode_step (src_dir, proc_folder = proc_folder, keep_acq_numb= 10)

The pre processing sequence will not change. It will be the same as the one we used for the reference. Except if there is some averaging requirements

In [ ]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(exp_test_param,['threshold'], [0])

In [ ]:
exp_test_dict = pre_processing_step (exp_test_dict, pre_proc_seq, param_dict = exp_test_param)

### Processing sequence and variables (scrubbing and SBKG)

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_exp_full.fits')
add_to_dict(exp_test_param,['BB_mask'], [BB_mask])

In [ ]:
exp_test_dict = processing_step (exp_test_dict, proc_seq, param_dict = exp_test_param)

### Displacement correction
For this specific experiment, there is just a y-axis displacement. Therefore, `dof = ['ty']`

We also knew previously the ROIs of the regions for the correction `reg_rois_list`. Otherwise, we would have to select them in an extra step

In [ ]:
reg_img = get_img(src_dir + '/ref_reg_img.fits')
reg_rois_list = [([22, 350, 30, 135], 'v'), ([418, 350, 30, 135], 'v')]

img = extract_img_dict(exp_test_dict, proc_folder[0], img_number = 15)
show_img_rois(img[0], dr = [(reg_rois_list, 'yellow')])

check_ref = extract_img_dict(ref_test_dict, ref_folder[0], img_number = 50)

In [ ]:
disp = float(0.566)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
img_reg_corr, M = img_registration (img, reg_img, reg_rois_list = reg_rois_list, dof=['ty'],M=M)
show_img(img_reg_corr[0]/check_ref[0], cmap = 'gray')
print(M)

In [ ]:
proc_seq = [image_registration_dict]

In [ ]:
add_to_dict(exp_test_param,['M', 'dof'], [M, ['ty']])

In [ ]:
exp_test_dict_after_reg = processing_step (exp_test_dict, proc_seq, param_dict = exp_test_param)

In [ ]:
dst_dir_test = dst_dir + '/exp1XX_after_registration'
saving_step (exp_test_dict_after_reg, dst_dir_test, img_name = 'exp102_reg_testing')

In [ ]:
exp_test_dict = processing_step (exp_test_dict, proc_seq, param_dict = exp_test_param)

### TFC and final transission

In [ ]:
proc_seq = [TFC_correction_dict]

In [ ]:
add_to_dict(exp_test_param,['nca', 'use_ref', 'ref_dict'], [nca, True, ref_test_dict])

In [ ]:
exp_test_dict = processing_step (exp_test_dict, proc_seq, param_dict = exp_test_param)

In [ ]:
dst_dir_test = dst_dir + '/exp1XX_avg_test'
saving_step (exp_test_dict, dst_dir_test, img_name = 'exp102_avg_testing')

## Reference Full Processing

In [ ]:
ref_dict, ref_param = reading_step (src_dir, proc_folder = ref_folder)

In [ ]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(ref_param,['threshold'], [0])

In [ ]:
ref_dict = pre_processing_step (ref_dict, pre_proc_seq, param_dict = ref_param)

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
add_to_dict(ref_param,['BB_mask'], [BB_mask])

In [ ]:
ref_dict = processing_step (ref_dict, proc_seq, param_dict = ref_param)

In [ ]:
dst_dir_test = dst_dir + '/00_Ref_dict_processed'
saving_step (ref_dict, dst_dir_test, img_name = 'so_ref_image')

## Experiment Full Averaged Processing

In [ ]:
exp_dict, exp_param = reading_step (src_dir, proc_folder = proc_folder)

In [ ]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(exp_param,['threshold'], [0])

In [ ]:
exp_dict = pre_processing_step (exp_dict, pre_proc_seq, param_dict = exp_param)

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict, image_registration_dict, TFC_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_exp_full.fits')
add_to_dict(exp_param,['BB_mask', 'nca', 'use_ref', 'M', 'dof', 'ref_dict'], 
                        [BB_mask, nca, True, M, ['ty'], ref_dict])

In [ ]:
exp_dict = processing_step (exp_dict, proc_seq, param_dict = exp_param)

In [ ]:
saving_step (exp_dict, dst_dir, img_name = 'avg_transmission_img')

## Experiment Full  Processing

In [ ]:
proc_folder = ['02_exp102_00']
exp_dict, exp_param = reading_step (src_dir, proc_folder = proc_folder)
pre_proc_seq = [outlier_removal]
add_to_dict(exp_param,['threshold'], [0])

In [ ]:
exp_dict = pre_processing_step (exp_dict, pre_proc_seq, param_dict = exp_param)

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict, image_registration_dict, TFC_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_exp_full.fits')
disp = float(0.5666)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
add_to_dict(exp_param,['BB_mask', 'nca', 'use_ref', 'M', 'dof', 'ref_dict'], 
                        [BB_mask, nca, True, M, ['ty'], ref_dict])

In [ ]:
exp_dict = processing_step (exp_dict, proc_seq, param_dict = exp_param)

In [ ]:
saving_step (exp_dict, dst_dir, img_name = 'transmission_img')

In [ ]:
var_1 = [1,2,3,4]
%store var_1

In [ ]:
proc_folder = ['02_exp102_01']
disp = float(0.64335)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
add_to_dict(exp_param,['M', 'start_acq_numb'],[M, 40])
exp_dict, _ = reading_step (src_dir, proc_folder = proc_folder)
exp_dict = pre_processing_step (exp_dict, pre_proc_seq, param_dict = exp_param)
exp_dict = processing_step (exp_dict, proc_seq, param_dict = exp_param)
saving_step (exp_dict, dst_dir, img_name = 'transmission_img')

In [ ]:
proc_folder = ['02_exp102_02']
disp = float(0.70334)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
add_to_dict(exp_param,['M', 'start_acq_numb'],[M, 80])
exp_dict, _ = reading_step (src_dir, proc_folder = proc_folder)
exp_dict = pre_processing_step (exp_dict, pre_proc_seq, param_dict = exp_param)
exp_dict = processing_step (exp_dict, proc_seq, param_dict = exp_param)
saving_step (exp_dict, dst_dir, img_name = 'transmission_img')

In [ ]:
proc_folder = ['02_exp102_03']
disp = float(0.75)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
add_to_dict(exp_param,['M', 'start_acq_numb'],[M, 120])
exp_dict, _ = reading_step (src_dir, proc_folder = proc_folder)
exp_dict = pre_processing_step (exp_dict, pre_proc_seq, param_dict = exp_param)
exp_dict = processing_step (exp_dict, proc_seq, param_dict = exp_param)
saving_step (exp_dict, dst_dir, img_name = 'transmission_img')

In [ ]:
proc_folder = ['02_exp102_04']
disp = float(0.89667)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
add_to_dict(exp_param,['M', 'start_acq_numb'],[M, 160])
exp_dict, _ = reading_step (src_dir, proc_folder = proc_folder)
exp_dict = pre_processing_step (exp_dict, pre_proc_seq, param_dict = exp_param)
exp_dict = processing_step (exp_dict, proc_seq, param_dict = exp_param)
saving_step (exp_dict, dst_dir, img_name = 'transmission_img')